WEEK 4 & 5

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import os

In [3]:
#file path
file_path = '/content/drive/MyDrive/IDX exchange'

#load datasets
sold = pd.read_csv(os.path.join(file_path, "CRMLSSold_WithMortgageRates.csv"),low_memory=False)

listings = pd.read_csv(os.path.join(file_path, "CRMLSListing_WithMortgageRates.csv"),low_memory=False)

In [4]:
sold_rows_before = len(sold)
listings_rows_before = len(listings)

print(f'Sold rows before cleaning: {sold_rows_before}')
print(f'Listings rows before cleaning: {listings_rows_before}')

Sold rows before cleaning: 433592
Listings rows before cleaning: 567549


In [5]:
# convert date fields to datetime format

date_columns = ['CloseDate','PurchaseContractDate','ListingContractDate','ContractStatusChangeDate']

for col in date_columns:
    if col in sold.columns:
        sold[col] = pd.to_datetime(sold[col], errors='coerce')

    if col in listings.columns:
        listings[col] = pd.to_datetime(listings[col], errors='coerce')

numeric_columns = ['ClosePrice','LivingArea','DaysOnMarket','BedroomsTotal','BathroomsTotalInteger']

for col in numeric_columns:
    if col in sold.columns:
        sold[col] = pd.to_numeric(sold[col], errors='coerce')

    if col in listings.columns:
        listings[col] = pd.to_numeric(listings[col], errors='coerce')

print('\nNumeric data types:')
print(sold[numeric_columns].dtypes)



Numeric data types:
ClosePrice               float64
LivingArea               float64
DaysOnMarket               int64
BedroomsTotal            float64
BathroomsTotalInteger    float64
dtype: object


In [6]:
#removes unecessary columns
columns_to_remove = [
    "Unnamed: 0"
]

sold.drop(columns=columns_to_remove, errors="ignore", inplace=True)
listings.drop(columns=columns_to_remove, errors="ignore", inplace=True)

#removes invalid numeric records

sold = sold[
    (sold['ClosePrice'] > 0) & (sold['LivingArea'] > 0) &
    (sold['DaysOnMarket'] >= 0) & (sold['BedroomsTotal'] >= 0) &
    (sold['BathroomsTotalInteger'] >= 0)
]

listings = listings[
    (listings['LivingArea'] > 0) & (listings['DaysOnMarket'] >= 0)&
    (listings['BedroomsTotal'] >= 0) & (listings['BathroomsTotalInteger'] >= 0)
]

In [7]:
#handles missing values

sold = sold.dropna(subset=['ClosePrice', 'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger'])
listings = listings.dropna(subset=['ListPrice', 'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger'])

listings = listings[(listings['ListPrice'] > 0) & (listings['LivingArea'] > 0) &
 (listings['DaysOnMarket'] >= 0) & (listings['BedroomsTotal'] >= 0) &
 (listings['BathroomsTotalInteger'] >= 0)]


In [8]:
# date consistency check
sold['listing_after_close_flag'] = sold['ListingContractDate'] > sold['CloseDate']
sold['purchase_after_close_flag'] = sold['PurchaseContractDate'] > sold['CloseDate']
sold['negative_timeline_flag'] = sold['PurchaseContractDate'] < sold['ListingContractDate']

#geographic data check
sold['missing_coordinates_flag'] = sold['Latitude'].isna() | sold['Longitude'].isna()
sold['zero_coordinate_flag'] = (sold['Latitude'] == 0) | (sold['Longitude'] == 0)
sold['positive_longitude_flag'] = sold['Longitude'] > 0

sold['invalid_coordinate_flag'] = (
    (sold['Latitude'] < 32) |
    (sold['Latitude'] > 42) |
    (sold['Longitude'] < -125) |
    (sold['Longitude'] > -114)
)

In [10]:
print('\nData Types')
print(sold.dtypes)

print(f'\nSold rows after cleaning: {len(sold)}')
print(f'Listings rows after cleaning: {len(listings)}')

print('\nDate Consistency Flags')
print(f'Listing after Close: {sold["listing_after_close_flag"].sum()}')
print(f'Purchase after Close: {sold["purchase_after_close_flag"].sum()}')
print(f'Negative Timeline: {sold["negative_timeline_flag"].sum()}')

print('\nGeographic Quality Summary')
print(f'Missing Coordinates: {sold["missing_coordinates_flag"].sum()}')
print(f'Zero Coordinates: {sold["zero_coordinate_flag"].sum()}')
print(f'Positive Longitude: {sold["positive_longitude_flag"].sum()}')
print(f'Invalid Coordinates: {sold["invalid_coordinate_flag"].sum()}')

#saving cleaned datasets

sold.to_csv(os.path.join(file_path, 'CRMLSSold_Cleaned.csv'), index=False)
listings.to_csv(os.path.join(file_path, 'CRMLSListing_Cleaned.csv'), index=False)

print('\nCleaned datasets saved successfully.')


Data Types
BuyerAgentAOR               object
ListAgentAOR                object
Flooring                    object
ViewYN                      object
WaterfrontYN                object
                             ...  
negative_timeline_flag        bool
missing_coordinates_flag      bool
zero_coordinate_flag          bool
positive_longitude_flag       bool
invalid_coordinate_flag       bool
Length: 93, dtype: object

Sold rows after cleaning: 401412
Listings rows after cleaning: 566478

Date Consistency Flags
Listing after Close: 75
Purchase after Close: 256
Negative Timeline: 248

Geographic Quality Summary
Missing Coordinates: 11521
Zero Coordinates: 24
Positive Longitude: 33
Invalid Coordinates: 99

Cleaned datasets saved successfully.
